In [1]:
# CELL 1 — Install Dependencies
import subprocess
import sys

packages = [
    "fastapi",
    "uvicorn[standard]",
    "pydantic",
    "nest_asyncio",
    "pyngrok"
]

print("📦 Memeriksa dependencies...")
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    except Exception as e:
        print(f"Peringatan install {pkg}: {e}")
print("✅ Semua dependensi siap!")


📦 Memeriksa dependencies...
✅ Semua dependensi siap!


In [2]:
# CELL 2 — Imports & Konfigurasi Global
import os
import re
import html
import unicodedata
import warnings
from enum import Enum
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")

# ── Enum Parameter Klasifikasi & Aksi Sistem ─────────────────────────
class CommentCategory(str, Enum):
    POSITIF = "Kritik & Saran Positif"
    ANEH_KASAR = "Comment aneh-aneh (komentar yang berbahasa kasar)"
    NEGATIF = "Kritik & Saran Negatif"


class ModerationAction(str, Enum):
    DELIVERED_TO_DASHBOARD = "DELIVERED_TO_DASHBOARD"
    AUTO_REMOVED = "AUTO_REMOVED"


class SentimentType(str, Enum):
    POSITIVE = "POSITIVE"
    NEGATIVE = "NEGATIVE"
    TOXIC = "TOXIC"


print("✅ Enum parameter klasifikasi berhasil dikonfigurasi:")
for cat in CommentCategory:
    print(f" • {cat.value}")


✅ Enum parameter klasifikasi berhasil dikonfigurasi:
 • Kritik & Saran Positif
 • Comment aneh-aneh (komentar yang berbahasa kasar)
 • Kritik & Saran Negatif


In [ ]:
# CELL 3 — Kamus & Lexicon Knowledge Base Bahasa Indonesia & English
# (Slang, Makian Kasar, Pola Sensor Bintang, Aspek, dan Leksikon Sentimen)

# 1. Kamus Slang & Singkatan Gaul (Bahasa Indonesia & English)
SLANG_MAP = {
    # ── Negasi & Penolakan (Indonesian & English) ─────────────────────
    "ga": "tidak", "gak": "tidak", "gk": "tidak", "g": "tidak", "kaga": "tidak",
    "kagak": "tidak", "tdk": "tidak", "ngga": "tidak", "nggak": "tidak", "nda": "tidak",
    "ndak": "tidak", "ora": "tidak", "ogah": "tidak", "enggak": "tidak", "engga": "tidak",
    "tak": "tidak", "bkn": "bukan", "gajadi": "tidak jadi", "gabisa": "tidak bisa",
    "gbs": "tidak bisa", "gamau": "tidak mau", "gkmau": "tidak mau", "gapapa": "tidak apa-apa",
    "gatau": "tidak tahu", "gktau": "tidak tahu",
    "dont": "tidak", "don't": "tidak", "doesnt": "tidak", "doesn't": "tidak",
    "didnt": "tidak", "didn't": "tidak", "wont": "tidak", "won't": "tidak",
    "cant": "tidak bisa", "can't": "tidak bisa", "isnt": "tidak", "isn't": "tidak",
    "arent": "tidak", "aren't": "tidak",

    # ── Keterangan Derajat / Penguat / Intensifier (Indonesian & English)
    "bgt": "banget", "bgtss": "banget", "bgtt": "banget", "bngtt": "banget", "bnget": "banget",
    "bgtz": "banget", "buanget": "banget", "bingits": "banget", "beut": "banget",
    "beneran": "sungguh", "bener2": "sungguh", "tenan": "sungguh",
    "skali": "sekali", "skl": "sekali", "amat": "sangat", "sangatt": "sangat",
    "parah": "parah", "parahh": "parah", "pisan": "sekali", "pol": "banget", "poll": "banget",
    "rly": "really", "sooo": "so", "supeer": "super",

    # ── Pronomina & Sapaan Pelanggan/Penjual (Indonesian & English) ───
    "sy": "saya", "aq": "saya", "aku": "saya", "ak": "saya", "gw": "saya", "gue": "saya",
    "gua": "saya", "gwa": "saya", "gwe": "saya", "w": "saya", "kmi": "kami",
    "lu": "kamu", "lo": "kamu", "loe": "kamu", "lw": "kamu", "ente": "kamu",
    "km": "kamu", "kmu": "kamu", "sampeyan": "kamu", "u": "kamu", "ur": "kamu",
    "min": "admin", "mimin": "admin", "adm": "admin", "seller": "penjual", "seler": "penjual",
    "kak": "kakak", "kaka": "kakak", "sis": "kakak", "sist": "kakak", "sista": "kakak",
    "gan": "agan", "bro": "abang", "bang": "abang", "om": "paman",

    # ── Kata Hubung, Partikel & Preposisi (Indonesian & English) ──────
    "tp": "tapi", "tpi": "tapi", "ttpi": "tapi", "ttp": "tetap", "tetep": "tetap",
    "dgn": "dengan", "dg": "dengan",
    "utk": "untuk", "untk": "untuk", "bwt": "buat",
    "dr": "dari", "dri": "dari",
    "krn": "karena", "krena": "karena", "karna": "karena", "krna": "karena", "coz": "karena",
    "cuz": "karena", "bcoz": "karena",
    "jd": "jadi", "jdi": "jadi",
    "kalo": "kalau", "kl": "kalau", "klo": "kalau", "klu": "kalau",
    "trs": "terus", "trus": "terus", "truss": "terus",
    "sm": "sama", "sma": "sama",
    "dlm": "dalam",
    "sblm": "sebelum", "sebelom": "sebelum", "b4": "sebelum",
    "stlh": "setelah", "ssdh": "sesudah",
    "bs": "bisa", "bsa": "bisa",
    "jg": "juga", "jga": "juga",
    "aja": "saja", "aj": "saja", "ajh": "saja", "doang": "saja", "thok": "saja",
    "dong": "dong", "donk": "dong", "dunk": "dong",
    "nih": "ini", "ni": "ini", "nie": "ini",
    "tuh": "itu", "tu": "itu",
    "udh": "sudah", "uda": "sudah", "udah": "sudah", "sdh": "sudah", "sudh": "sudah",
    "blm": "belum", "blom": "belum", "bleum": "belum",
    "lg": "lagi", "lgi": "lagi",
    "emg": "memang", "emang": "memang",
    "knp": "kenapa", "knapa": "kenapa", "kenapah": "kenapa",
    "gmn": "bagaimana", "gmana": "bagaimana", "gimana": "bagaimana",
    "gtu": "begitu", "gitu": "begitu",
    "gni": "begini", "gini": "begini",
    "trnyata": "ternyata", "ternyta": "ternyata", "taunya": "ternyata",
    "tbh": "honestly", "idk": "tidak tahu", "imo": "in my opinion", "imho": "in my opinion",
    "fyi": "fyi", "asap": "asap", "thru": "through",

    # ── Makanan, Rasa & Kualitas ──────────────────────────────────────
    "bgs": "bagus", "bgus": "bagus", "mantul": "mantap", "mantep": "mantap",
    "mantapp": "mantap", "mantabb": "mantap", "top": "mantap", "jos": "mantap",
    "josss": "mantap", "maknyus": "mantap", "ciamik": "mantap", "apik": "bagus",
    "delish": "delicious", "yumm": "yummy", "fav": "favorite", "fave": "favorite",
    "krg": "kurang", "kurg": "kurang", "kurng": "kurang",
    "bnyk": "banyak", "byk": "banyak", "bejibun": "banyak",
    "dikit": "sedikit", "sdkt": "sedikit", "dkit": "sedikit", "seiprit": "sedikit",
    "mkn": "makan", "mknn": "makanan", "makann": "makanan",
    "minum": "minuman", "mnm": "minuman", "mnmn": "minuman",
    "hrg": "harga", "mrh": "murah", "murmer": "murah", "murce": "murah",
    "mhl": "mahal", "mehong": "mahal",
    "asin": "asin", "keasinan": "asin", "hambar": "hambar", "anyep": "hambar", "cemplang": "hambar",
    "basi": "basi", "basii": "basi", "mambu": "basi",
    "bau": "bau", "apek": "bau", "amis": "bau", "tengik": "bau",
    "mentah": "mentah", "mengkal": "mentah",
    "gosong": "gosong", "hangus": "gosong", "tutong": "gosong",
    "pds": "pedas", "pedes": "pedas", "kepedesan": "pedas",
    "pait": "pahit", "kepaitan": "pahit",
    "kemanisan": "manis",
    "alot": "keras", "liat": "keras", "kras": "keras",
    "dingin": "dingin", "adem": "dingin",
    "anget": "hangat",
    "renyah": "renyah", "crispy": "renyah", "krispi": "renyah", "kriuk": "renyah",
    "seger": "segar", "fresh": "segar",

    # ── Kemasan, Fisik & Kebersihan ───────────────────────────────────
    "packing": "kemasan", "packaging": "kemasan", "bungkus": "kemasan", "bungkusnya": "kemasan",
    "packingan": "kemasan", "wadah": "kemasan", "pic": "picture",
    "ancur": "hancur", "remuk": "hancur", "penyok": "hancur", "peyot": "hancur",
    "tumpah": "tumpah", "bleber": "tumpah", "bocor": "bocor", "rembes": "bocor",
    "robek": "robek", "sobek": "robek",
    "kotor": "kotor", "jorok": "kotor", "kumel": "kotor",
    "rapi": "rapi", "rapih": "rapi", "rapet": "rapi",
    "higienis": "bersih", "resik": "bersih",

    # ── Pelayanan, Waktu & Sikap ──────────────────────────────────────
    "ramah": "ramah", "baek": "ramah", "helpful": "ramah",
    "judes": "judes", "jutek": "judes", "jutex": "judes", "galak": "galak", "cuek": "cuek", "ketus": "judes",
    "cepet": "cepat", "cpt": "cepat", "fast": "cepat", "kenceng": "cepat", "gesit": "cepat", "satset": "cepat",
    "lama": "lambat", "lemot": "lambat", "lelet": "lambat", "llet": "lambat", "ngaret": "lambat",
    "gercep": "responsif", "tanggap": "responsif",
    "dibalas": "dibalas", "dibales": "dibalas", "direply": "dibalas", "direspon": "dibalas",
    "diabaiin": "diabaikan", "dicuekin": "diabaikan", "dikacangin": "diabaikan",

    # ── Pesanan & Logistik ────────────────────────────────────────────
    "pesen": "pesan", "pesenan": "pesanan", "orderan": "pesanan", "ordran": "pesanan", "order": "pesanan",
    "ongkir": "ongkir", "ongkos": "ongkir",
    "kirim": "pengiriman", "kiriman": "pengiriman", "antar": "pengiriman", "delivery": "pengiriman",
    "nyampe": "sampai", "sampe": "sampai",
    "kurir": "kurir", "ojol": "kurir", "driver": "kurir",

    # ── Ulasan, Kepuasan & Saran ──────────────────────────────────────
    "recomended": "rekomendasi", "rekomen": "rekomendasi", "recom": "rekomendasi", "recommended": "rekomendasi",
    "worth": "layak", "worthit": "layak", "worthed": "layak",
    "nyesel": "menyesal", "kecewa": "kecewa", "kapok": "kapok", "zonk": "zonk", "rugi": "rugi",
    "nipu": "menipu", "boong": "bohong",
    "bikin": "membuat", "bkin": "membuat",
    "tlg": "tolong", "pls": "tolong", "plz": "tolong", "pliss": "tolong", "tlng": "tolong", "mohon": "tolong",
    "mks": "terima kasih", "makasi": "terima kasih", "makasih": "terima kasih",
    "tq": "terima kasih", "thx": "terima kasih", "thanks": "terima kasih", "ty": "terima kasih",
    "next": "selanjutnya", "kedepan": "kedepannya", "kdepan": "kedepannya", "kedepannya": "kedepannya",
    "adain": "sediakan", "sediain": "sediakan",
    "tambahin": "ditambah", "benerin": "perbaiki", "perbaiki": "perbaiki",
    "estetik": "estetik", "aesthetic": "estetik"
}

# 2. Kamus Kata Kasar & Makian (Indonesian & English Toxic Lexicon)
ABUSIVE_WORDS = {
    # Hewan sebagai makian
    "anjing", "anj1ng", "anjink", "anying", "anjeng", "ajg", "asw", "asu", "asuu",
    "babi", "b4b1", "bb1", "monyet", "kunyuk", "bajing", "celeng",
    "anjir", "bjir", "njir", "njirs", "njirr",

    # Makian kasar, umpatan & caci maki daerah
    "bangsat", "b@ngs@t", "b4ngs4t", "bajingan", "b4j1ng4n", "kampret", "brengsek", "keparat",
    "jancuk", "jancok", "dancok", "dancuk", "cok", "cuk", "ancuk", "ancok",
    "pantek", "pantak", "puki", "pukimak", "pukima", "kimak", "bodat", "bedebah",
    "bangke", "bangkai", "tai", "taek", "tae", "taik", "setan", "iblis", "dajjal", "bejat",
    "sialan", "biadab", "laknat", "mampus", "modar", "modyar", "mati aja", "matio",

    # Organ reproduksi & seksual kasar (ID & EN)
    "kontol", "k0nt0l", "kntil", "kntl", "kontl", "kntol",
    "memek", "mmk", "m3m3k", "jembut", "jmbt",
    "peler", "paler", "pelir", "itil", "titit", "tetek", "toket", "pepek", "tempik",
    "ngentot", "ngentod", "gentot", "ngewe", "ewe", "crot", "sange", "sangean",
    "lonte", "lont3", "pelacur", "jablay", "jabl4y", "perek", "sundal", "pecun", "bispak",
    "bencong", "banci", "dick", "pussy", "cock", "cocksucker", "prick", "twat", "wanker",

    # Cacian mental/intelektual kasar (ID & EN)
    "goblok", "g0bl0k", "gblk", "goblog", "goblokk", "tolol", "t0l0l", "tll", "bego", "bgok", "begok",
    "idiot", "pekok", "pekokk", "autis", "dungu", "otak udang", "bloon", "oon", "geblek", "koplak",
    "moron", "stupid", "retard", "retarded", "dumbass", "dipshit", "jackass",

    # Bahasa Inggris kasar, vulgar & umpatan
    "hell", "fuck", "fucking", "fucker", "fck", "motherfucker", "bitch", "bitches", "b!tch",
    "asshole", "bastard", "shit", "bullshit", "cunt", "slut", "whore", "douchebag", "scumbag",
    "stfu", "wtf"
}

# 3. Pola Regex untuk Makian Bertanda Bintang / Sensor Simbol (Indonesian & English)
MASKED_ABUSIVE_PATTERNS = [
    # bajingan & variasinya
    r'(?<!\w)b+a*j+[a-z]*[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)b+[\*\@\#\$\%\&\!\~\-\_]+j+i*n*g*a*n*(?!\w)',
    # anjing & variasinya
    r'(?<!\w)a+n*j*[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)a+[\*\@\#\$\%\&\!\~\-\_]+n*j+[a-z]*(?!\w)',
    # babi & variasinya
    r'(?<!\w)b+[\*\@\#\$\%\&\!\~\-\_]+[i|1]+(?!\w)',
    r'(?<!\w)b+[\*\@\#\$\%\&\!\~\-\_]+b+[i|1|\*]+(?!\w)',
    r'(?<!\w)b+a+[\*\@\#\$\%\&\!\~\-\_]+[i|1|\*]*(?!\w)',
    r'(?<!\w)b+[\*\@\#\$\%\&\!\~\-\_]{3,}(?!\w)',
    # kontol & variasinya
    r'(?<!\w)k+[\*\@\#\$\%\&\!\~\-\_]+n+t+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    r'(?<!\w)k+[\*\@\#\$\%\&\!\~\-\_]{2,}l+(?!\w)',
    r'(?<!\w)k+o+n+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)k+o*n*t+[\*\@\#\$\%\&\!\~\-\_]+l*(?!\w)',
    # goblok & variasinya
    r'(?<!\w)g+[\*\@\#\$\%\&\!\~\-\_]+b+l+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    r'(?<!\w)g+[\*\@\#\$\%\&\!\~\-\_]{2,}k+(?!\w)',
    r'(?<!\w)g+o+b+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)g+o*b*l+[\*\@\#\$\%\&\!\~\-\_]+k*(?!\w)',
    # bangsat & variasinya
    r'(?<!\w)b+[\*\@\#\$\%\&\!\~\-\_]+n+g+s+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    r'(?<!\w)b+a+n+g+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)b+a*n*g*s+[\*\@\#\$\%\&\!\~\-\_]+t*(?!\w)',
    # memek & variasinya
    r'(?<!\w)m+[\*\@\#\$\%\&\!\~\-\_]+m+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    r'(?<!\w)m+e+[\*\@\#\$\%\&\!\~\-\_]+k+(?!\w)',
    # tolol & variasinya
    r'(?<!\w)t+[\*\@\#\$\%\&\!\~\-\_]+l+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    r'(?<!\w)t+o+[\*\@\#\$\%\&\!\~\-\_]+l+(?!\w)',
    # jancuk/jancok & variasinya
    r'(?<!\w)j+a*n*c+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)j+a+n+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)j+[\*\@\#\$\%\&\!\~\-\_]+n+c+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    # pantek & variasinya
    r'(?<!\w)p+a*n*t+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)p+[\*\@\#\$\%\&\!\~\-\_]+n+t+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    # kampret & variasinya
    r'(?<!\w)k+a*m*p+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)k+[\*\@\#\$\%\&\!\~\-\_]+m+p+r+[a-z\*\@\#\$\%\&\!\~\-\_]*(?!\w)',
    # peler, jembut, lonte
    r'(?<!\w)p+[\*\@\#\$\%\&\!\~\-\_]+l+e*r+(?!\w)',
    r'(?<!\w)j+[\*\@\#\$\%\&\!\~\-\_]+m+b+u*t+(?!\w)',
    r'(?<!\w)l+[\*\@\#\$\%\&\!\~\-\_]+n+t+e*(?!\w)',
    # Bahasa Inggris bertanda sensor (fuck, bitch, shit, asshole, dick, etc)
    r'(?<!\w)f+[\*\@\#\$\%\&\!\~\-\_]+c*k+[a-z]*(?!\w)',
    r'(?<!\w)f+u+[\*\@\#\$\%\&\!\~\-\_]+k+(?!\w)',
    r'(?<!\w)f+[\*\@\#\$\%\&\!\~\-\_]{2,}(?!\w)',
    r'(?<!\w)b+[\*\@\#\$\%\&\!\~\-\_]+t*c*h+[a-z]*(?!\w)',
    r'(?<!\w)b+i+[\*\@\#\$\%\&\!\~\-\_]+h+(?!\w)',
    r'(?<!\w)s+h+[\*\@\#\$\%\&\!\~\-\_]+t+(?!\w)',
    r'(?<!\w)s+[\*\@\#\$\%\&\!\~\-\_]+i+t+(?!\w)',
    r'(?<!\w)a+[\*\@\#\$\%\&\!\~\-\_]+s+h+o*l*e*(?!\w)',
    r'(?<!\w)a+s+s+[\*\@\#\$\%\&\!\~\-\_]+[a-z]*(?!\w)',
    r'(?<!\w)d+[\*\@\#\$\%\&\!\~\-\_]+c*k+(?!\w)',
    r'(?<!\w)p+[\*\@\#\$\%\&\!\~\-\_]+s+s+y+(?!\w)',
    r'(?<!\w)c+[\*\@\#\$\%\&\!\~\-\_]+n*t+(?!\w)',
]

# 4. Kamus Aspek Review untuk Analisis Dashboard Toko (Bilingual: ID & EN)
ASPECT_KEYWORDS = {
    "Rasa & Kualitas": [
        # Bahasa Indonesia
        "rasa", "rasanya", "cita rasa", "tekstur", "bumbu", "bumbunya", "enak", "enak banget",
        "mantap", "lezat", "sedap", "nagih", "nikmat", "maknyus", "asin", "keasinan", "hambar",
        "cemplang", "anyep", "manis", "kemanisan", "pahit", "pait", "kepaitan", "pedas", "pedes",
        "kepedesan", "gurih", "gurihnya", "micin", "basi", "mambu", "bau", "amis", "tengik",
        "apek", "bacin", "mentah", "mengkal", "gosong", "hangus", "tutong", "dingin", "anget",
        "hangat", "panas", "renyah", "crispy", "krispi", "garing", "kriuk", "alot", "keras",
        "lembut", "empuk", "juicy", "segar", "fresh", "matang", "mateng", "aromanya", "wangi",
        "harum", "oily", "berminyak", "enek", "eneg", "creamy", "pekat", "encer", "kopi", "teh",
        # English
        "taste", "tasted", "flavor", "flavour", "flavorful", "delicious", "tasty", "yummy", "delish",
        "savory", "savoury", "salty", "oversalted", "bland", "tasteless", "sweet", "bitter", "spicy",
        "sour", "spoiled", "rotten", "stale", "smelly", "stinky", "raw", "undercooked", "burnt",
        "burned", "overcooked", "cold", "warm", "hot", "crunchy", "tough", "tender",
        "cooked", "seasoning", "greasy", "coffee", "tea", "drink", "drinks", "beverage",
        "aroma", "aromas", "scent", "food", "dish", "meal", "chicken", "beef"
    ],
    "Porsi & Takaran": [
        # Bahasa Indonesia
        "porsi", "porsinya", "takaran", "dikit", "sedikit", "pelit", "banyak", "melimpah",
        "kenyang", "pas", "jumbo", "mini", "ukuran", "ukurannya", "size", "potongan", "irisan",
        "isi", "isinya", "topping", "toping", "nasi", "lauk", "daging", "sayur", "tebel",
        "tebal", "tipis", "lebar", "seiprit", "nanggung", "cukup",
        # English
        "portion", "portions", "amount", "quantity", "serving", "huge", "big", "large", "small",
        "tiny", "little", "filling", "stuffed", "slice", "slices", "piece", "pieces", "meat",
        "rice", "thick", "thin"
    ],
    "Kemasan & Kebersihan": [
        # Bahasa Indonesia
        "kemasan", "packing", "packaging", "packingan", "bungkus", "bungkusnya", "wadah", "kotak",
        "box", "cup", "gelas", "plastik", "paper bag", "paperbag", "kantong", "tutup", "seal",
        "sealer", "bubble wrap", "tumpah", "bocor", "rembes", "robek", "sobek", "hancur", "ancur",
        "remuk", "penyok", "peyot", "rapi", "rapih", "rapet", "aman", "safety", "higienis",
        "hygienic", "bersih", "kotor", "debu", "lalat", "rambut", "kecoak", "jorok", "resik",
        "steril", "sendok", "garpu", "sedotan", "tisu", "tissue",
        # English
        "package", "pack", "container", "spilled", "spilt", "leak", "leaking", "leaked", "torn",
        "crushed", "damaged", "neat", "tidy", "secure", "clean", "cleanliness", "hygiene",
        "dirty", "filthy", "dust", "hair", "cockroach", "spoon", "fork", "straw", "napkin"
    ],
    "Pelayanan & Tenant": [
        # Bahasa Indonesia
        "pelayanan", "service", "layanan", "penjual", "seller", "admin", "mimin", "kasir",
        "barista", "waiter", "pelayan", "staff", "staf", "karyawan", "owner", "owner-nya",
        "ramah", "sopan", "santun", "humble", "baik", "helpful", "judes", "jutek", "galak",
        "ketus", "cuek", "jutex", "respon", "response", "chat", "dibalas", "dibales",
        "slow respon", "fast respon", "gercep", "diabaikan", "tanggap", "komunikatif",
        "salah pesanan", "salah kirim", "kurang kirim", "salah menu", "tertukar",
        # English
        "customer service", "vendor", "shop", "store", "waitress", "employee", "polite",
        "kind", "attentive", "rude", "unfriendly", "reply", "replied", "slow response",
        "fast response", "ignored", "wrong item", "wrong order", "missing item"
    ],
    "Kecepatan & Pengiriman": [
        # Bahasa Indonesia
        "cepat", "cepet", "kilat", "gercep", "satset", "express", "lama", "lambat", "lelet",
        "ngaret", "nunggu", "antri", "antrean", "antrian", "nunggunya", "jam", "menit",
        "antar", "antarnya", "antarannya", "kurir", "driver", "ojol", "pengiriman", "kirim",
        "kiriman", "sampai", "nyampe", "durasi", "waktu tunggu", "delay",
        # English
        "speed", "swift", "slow", "late", "delayed", "wait", "waiting", "waited", "queue",
        "queueing", "hour", "hours", "minute", "minutes", "delivery", "courier", "rider",
        "shipping", "arrived", "arrival", "duration"
    ],
    "Harga & Nilai": [
        # Bahasa Indonesia
        "harga", "harganya", "price", "biaya", "murah", "murmer", "terjangkau", "ekonomis",
        "hemat", "mahal", "kemahalan", "mehong", "kantong", "worth", "worthit", "worthed",
        "layak", "sebanding", "sesuai", "padan", "promo", "diskon", "potongan harga",
        "voucher", "ongkir", "cashback", "pricey", "kemahalan",
        # English
        "pricing", "cost", "cheap", "affordable", "inexpensive", "budget", "expensive",
        "overpriced", "worth it", "worthwhile", "valuable", "value", "deal", "discount",
        "shipping fee"
    ]
}

# 5. Indikator Saran / Rekomendasi Konstruktif (Bilingual: ID & EN)
SUGGESTION_KEYWORDS = [
    # Bahasa Indonesia
    "saran", "sarannya", "masukan", "usul", "usulan", "rekomendasi", "request", "nasihat",
    "kritik", "feedback", "mungkin", "bisa", "bisakah", "boleh", "tolong", "mohon", "harap",
    "kiranya", "minta", "ada baiknya", "alangkah baiknya", "bagus kalau", "lebih baik",
    "enaknya", "sebaiknya", "seharusnya", "mending", "mendingan", "ke depannya", "kedepannya",
    "kedepan", "next", "next time", "berikutnya", "nanti", "nanti lagi", "lain kali", "perlu",
    "harus", "ditingkatkan", "tingkatkan", "tingkatin", "ditambah", "tambahkan", "tambahin",
    "diperbaiki", "perbaiki", "benerin", "dikurangi", "kurangi", "kurangin", "disesuaikan",
    "ganti", "diganti", "disediakan", "sediakan", "adain", "diadain", "perhatikan",
    "diperhatikan", "cek lagi", "lebih teliti", "diteliti", "dijaga", "variasi", "varian",
    "opsi", "menu baru", "pilihan", "alternatif", "inovasi", "coba",
    # English
    "suggest", "suggestion", "suggestions", "recommend", "recommendation", "input", "advice",
    "maybe", "perhaps", "could", "could you", "can you", "please", "kindly", "would be great if",
    "would be nice if", "better if", "hope", "hopefully", "should", "need to", "improve",
    "improved", "improvement", "enhance", "increase", "add", "added", "fix", "repair",
    "reduce", "provide", "consider", "option", "options", "variety", "variation", "variations",
    "alternative", "new menu", "in the future"
]

# 6. Sentiment Lexicon Positif (Bilingual: ID & EN)
POSITIVE_WORDS = {
    # Bahasa Indonesia
    "enak", "uenak", "lezat", "nikmat", "sedap", "maknyus", "gurih", "empuk", "lembut",
    "juicy", "renyah", "garing", "kriuk", "crispy", "krispi", "segar", "fresh", "hangat",
    "anget", "wangi", "harum", "nagih", "candu", "nagihin", "mantap", "mantul", "mantep",
    "jos", "josss", "puas", "suka", "senang", "favorit", "favorite", "cinta", "love",
    "rekomendasi", "rekomen", "recommended", "recomended", "top", "juara", "juaranya",
    "terbaik", "jempol", "keren", "bagus", "ciamik", "apik", "istimewa", "sempurna",
    "perfect", "hebat", "berkualitas", "premium", "ramah", "sopan", "santun", "humble",
    "baik", "tanggap", "responsif", "gercep", "helpful", "perhatian", "amanah", "jujur",
    "rapi", "rapih", "rapet", "bersih", "higienis", "hygienic", "resik", "steril",
    "estetik", "aesthetic", "cantik", "menarik", "aman", "safety", "cepat", "cepet",
    "kilat", "gesit", "satset", "express", "murah", "murmer", "terjangkau", "ekonomis",
    "hemat", "worth", "worthit", "worthed", "layak", "sebanding", "pas", "berkah", "untung",
    "langganan", "berlangganan", "repeat", "reorder", "oke", "ok", "sip", "lezatt",
    "enaknyoo", "memuaskan",
    # English
    "good", "great", "delicious", "tasty", "yummy", "delish", "savory", "flavorful",
    "amazing", "awesome", "fantastic", "excellent", "superb", "wonderful", "fabulous",
    "perfection", "tender", "warm", "hot", "satisfying", "satisfied", "loved", "favourite",
    "best", "attentive", "speedy", "prompt", "tidy", "spotless", "beautiful", "secure",
    "budget-friendly", "worthwhile", "generous", "happy", "pleased", "impressed", "solid", "super"
}

# 7. Sentiment Lexicon Negatif (Bilingual: ID & EN)
NEGATIVE_WORDS = {
    # Bahasa Indonesia
    "kurang", "kecewa", "menyesal", "nyesel", "kapok", "zonk", "rugi", "rugii",
    "hambar", "cemplang", "anyep", "asin", "keasinan", "kemanisan", "kepedesan",
    "pahit", "pait", "kepaitan", "basi", "basii", "mambu", "bau", "amis", "tengik",
    "apek", "bacin", "mentah", "mengkal", "gosong", "hangus", "tutong", "alot",
    "keras", "lembek", "benyek", "dingin", "adem", "oily", "berminyak", "enek",
    "eneg", "dikit", "sedikit", "pelit", "seiprit", "tumpah", "bocor", "rembes",
    "robek", "sobek", "hancur", "remuk", "rusak", "penyok", "peyot", "kotor",
    "jorok", "kumel", "debu", "lalat", "rambut", "kecoak", "lama", "lambat",
    "lelet", "ngaret", "mahal", "kemahalan", "mehong", "pricey", "judes", "jutek",
    "galak", "ketus", "cuek", "diabaikan", "tertukar", "penipu", "nipu", "bohong",
    "palsu", "buruk", "jelek", "ancur", "gagal", "kecele", "batal",
    "tidak ramah", "ga ramah", "tidak enak", "ga enak", "nggak enak",
    "tidak dibalas", "ga dibales", "diabaikan", "tidak niat", "ga niat",
    # English
    "bad", "worst", "worse", "terrible", "horrible", "awful", "poor", "poorly",
    "disappointed", "disappointing", "disappointment", "regret", "regretful", "regretted",
    "bland", "tasteless", "oversalted", "sour", "spoiled", "rotten", "stale", "smelly",
    "stinky", "raw", "undercooked", "burnt", "burned", "overcooked", "hard", "tough",
    "greasy", "sick", "nauseous", "small", "tiny", "little", "insufficient", "spilled",
    "spilt", "leaking", "leaked", "torn", "ripped", "crushed", "damaged", "broken",
    "dirty", "filthy", "messy", "disgusting", "gross", "unhygienic", "dusty", "slow",
    "delayed", "expensive", "overpriced", "rip-off", "ripoff", "rude", "impolite",
    "unfriendly", "unresponsive", "scam", "scammer", "fake", "dishonest", "ruined",
    "waste", "unacceptable"
}

print(f"✅ Kamus NLU selesai dimuat (Bilingual: Indonesia & English):")
print(f" • Total kata kasar dimonitor: {len(ABUSIVE_WORDS)} kata")
print(f" • Total kata slang dinormalisasi: {len(SLANG_MAP)} kata")
print(f" • Total pola sensoran kasar dimonitor: {len(MASKED_ABUSIVE_PATTERNS)} pola regex")
print(f" • Total kategori aspek ulasan: {len(ASPECT_KEYWORDS)} aspek")
print(f" • Total kata saran konstruktif: {len(SUGGESTION_KEYWORDS)} kata")
print(f" • Total kata sentimen positif: {len(POSITIVE_WORDS)} kata")
print(f" • Total kata sentimen negatif: {len(NEGATIVE_WORDS)} kata")


✅ Kamus NLU selesai dimuat:
 • Total kata kasar dimonitor: 74 kata
 • Total kata slang dinormalisasi: 108 kata
 • Total kategori aspek ulasan: 6 aspek


In [ ]:
# CELL 4 — Preprocessing & Leetspeak Normalizer Engine

def normalize_leetspeak(text: str) -> str:
    """
    Mengembalikan teks leetspeak / angka pengganti huruf ke karakter alfabet standar.
    Contoh: 'anj1ng' -> 'anjing', 'g0bl0k' -> 'goblok', 'k0nt0l' -> 'kontol', 'b.a.b.i' -> 'babi', 'b!tch' -> 'bitch'.
    Tanda seru di akhir kalimat (misal: 'enak!') tetap dipertahankan sebagai tanda baca.
    """
    # Hapus spasi atau titik pemisah sengaja antar huruf (misal: 'a.n.j.i.n.g' -> 'anjing')
    de_spaced = re.sub(r'(?<=\b[a-zA-Z0-9])[\.\-\_\s](?=[a-zA-Z0-9]\b)', '', text)

    # Ubah ! menjadi i hanya jika diikuti huruf (b!tch -> bitch), bukan tanda seru di akhir kata
    de_spaced = re.sub(r'!(?=[a-zA-Z])', 'i', de_spaced)

    leet_map = {
        '0': 'o', '1': 'i', '3': 'e', '4': 'a', '5': 's',
        '7': 't', '8': 'b', '@': 'a', '$': 's'
    }
    res = []
    for char in de_spaced:
        res.append(leet_map.get(char, char))
    return "".join(res)


def preprocess_text(text: str) -> Tuple[str, str]:
    """
    Pipeline NLU Preprocessing (Bilingual: ID & EN):
    1. HTML Unescape & Lowercasing
    2. Leetspeak Decoding
    3. Collapsing Karakter Berulang (misal: 'enaaaak' -> 'enak', 'soooo' -> 'so')
    4. Slang/Abbreviation Expansion ke bahasa formal (ID & EN)
    Mengembalikan: (raw_cleaned_text, normalized_text_for_analysis)
    """
    if not text:
        return "", ""

    # 1. Clean & Lowercase
    raw = html.unescape(text).strip()
    lowered = unicodedata.normalize('NFKD', raw).lower()

    # 2. Leetspeak Decoding
    normalized_leet = normalize_leetspeak(lowered)

    # 3. Collapse repeated chars (3x berturut-turut jadi 1)
    collapsed = re.sub(r'(.)\1{2,}', r'\1', normalized_leet)

    # 4. Tokenize & Slang Expansion
    tokens = re.findall(r"\b[\w'-]+\b", collapsed)
    expanded_tokens = [SLANG_MAP.get(tok, tok) for tok in tokens]
    expanded_text = " ".join(expanded_tokens)

    return raw, expanded_text


# Tes Cepat Preprocessing (Bilingual)
sample_id = "Woy penjual anj1ng, makanannya ga enak bgtss & kuah tumpaaah!!"
sample_en = "Really loved the coffee, amazing aroma and friendly barista!"
_, clean_id = preprocess_text(sample_id)
_, clean_en = preprocess_text(sample_en)
print("Indonesian Cleaned:", clean_id)
print("English Cleaned   :", clean_en)


Original: Woy penjual anj1ng, makanannya ga enak bgtss & kuah tumpaaah!!
Cleaned : woy penjual anjing makanannya tidak enak banget kuah tumpahii


In [5]:
# CELL 5 — Stage 1: Safety & Toxicity Gate (Filter Kasar / Aneh-aneh)

# Compile regex pattern untuk pencarian cepat & boundary-aware
escaped_abusive = [re.escape(w) for w in ABUSIVE_WORDS]
abusive_regex = re.compile(r'\b(' + '|'.join(escaped_abusive) + r')\b', re.IGNORECASE)
masked_regexes = [re.compile(p, re.IGNORECASE) for p in MASKED_ABUSIVE_PATTERNS]


def detect_profanity_and_toxicity(raw_text: str, preprocessed_text: str) -> Tuple[bool, List[str], str]:
    """
    Mengecek apakah komentar mengandung kata-kata kasar, makian, ujaran kebencian,
    sensoran leetspeak, atau konten toksik ('Comment aneh-aneh') baik bahasa Indonesia maupun English.
    """
    flagged = []
    lowered_raw = raw_text.lower()
    leet_normalized = normalize_leetspeak(lowered_raw)

    # 1. Cek kata kasar standar pada teks raw, leet, dan preprocessed
    for target in [lowered_raw, leet_normalized, preprocessed_text]:
        matches = abusive_regex.findall(target)
        for m in matches:
            clean_m = m.lower()
            if clean_m not in flagged:
                flagged.append(clean_m)

    # 2. Cek makian bertanda bintang / simbol (misal: baji***an, a**ing, k*nt*l, f*ck, b*tch, sh*t)
    for pattern in masked_regexes:
        matches = pattern.findall(lowered_raw)
        for m in matches:
            clean_m = m.lower()
            if clean_m not in flagged:
                flagged.append(f"sensoran ({clean_m})")

    # 3. Cek context-dependent insults (contoh: 'toko sampah', 'makanan sampah', 'najis', 'trash', 'piece of shit')
    if re.search(r'\b(sampah|najis|trash|garbage|scum)\b', preprocessed_text) or re.search(r'\b(piece of shit|go to hell|kill yourself|shut up)\b', lowered_raw):
        flagged.append("hinaan degradatif (sampah/najis/trash)")

    if flagged:
        reason = f"Terdeteksi kata-kata kasar / makian: {', '.join(flagged)}. Sesuai aturan moderasi, komentar otomatis di-remove."
        return True, flagged, reason

    return False, [], ""


# Uji Coba Safety Gate (Bilingual)
test_inputs = [
    "Penjual ramah sekali",
    "Woy babi lu jualan basi anj1ng!",
    "makanan baji***an nipu",
    "What the fuck is this food, stupid scammer!",
    "This store is full of assholes, trash service!",
    "F*ck you, scammer bitch!"
]
for txt in test_inputs:
    _, clean = preprocess_text(txt)
    toxic, flags, reason = detect_profanity_and_toxicity(txt, clean)
    print(f"Teks: '{txt}'")
    print(f" ↳ Kasar/Toxic: {toxic} | Flagged: {flags}")
    print("-" * 55)


Teks: 'Penjual ramah sekali'
 ↳ Kasar/Toxic: False | Flagged: []
--------------------------------------------------
Teks: 'Woy babi lu jualan basi anj1ng!'
 ↳ Kasar/Toxic: True | Flagged: ['babi', 'anj1ng']
--------------------------------------------------
Teks: 'makanan baji***an nipu'
 ↳ Kasar/Toxic: True | Flagged: ['sensoran (baji***an)']
--------------------------------------------------


In [6]:
# CELL 6 — Stage 2 & 3: Sentiment, Suggestion & Aspect Classifier Engine

def extract_aspects(text: str) -> List[str]:
    """Mengekstraksi aspek-aspek apa saja yang dibahas dalam komentar (Bilingual: ID & EN)."""
    found_aspects = []
    text_lower = text.lower()
    for aspect, keywords in ASPECT_KEYWORDS.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
                found_aspects.append(aspect)
                break
    return found_aspects


def classify_comment(
    text: str,
    comment_id: Optional[str] = None,
    tenant_id: Optional[str] = None,
    rating: Optional[float] = None
) -> Dict[str, Any]:
    """
    Pipeline Utama Klasifikasi Komentar Toko (Bilingual: Indonesia & English):
    - Stage 1: Safety Gate (Auto-remove toxic/profane/abusive)
    - Stage 2: Sentiment Lexicon & Negation Rule-based Scoring
    - Stage 3: Constructive Suggestion & Aspect Topic Extraction
    """
    raw_text, preprocessed_text = preprocess_text(text)

    # ── STAGE 1: Toxic & Profanity Filter ──
    is_toxic, flagged_words, toxic_reason = detect_profanity_and_toxicity(raw_text, preprocessed_text)
    if is_toxic:
        return {
            "comment_id": comment_id,
            "tenant_id": tenant_id,
            "original_text": raw_text,
            "cleaned_text": preprocessed_text,
            "category": CommentCategory.ANEH_KASAR.value,
            "action": ModerationAction.AUTO_REMOVED.value,
            "is_delivered_to_dashboard": False,
            "confidence_score": 0.98,
            "sentiment": SentimentType.TOXIC.value,
            "aspects": extract_aspects(preprocessed_text),
            "flagged_words": flagged_words,
            "moderation_reason": toxic_reason,
            "dashboard_badge": "DIBLOKIR / REMOVED"
        }

    # ── STAGE 2: Sentiment & Constructive Suggestion Classification ──
    pos_score = 0.0
    neg_score = 0.0
    suggestion_score = 0.0

    tokens = preprocessed_text.split()
    for tok in tokens:
        if tok in POSITIVE_WORDS:
            pos_score += 1.0
        if tok in NEGATIVE_WORDS:
            neg_score += 1.0
        if tok in SUGGESTION_KEYWORDS:
            suggestion_score += 0.8

    # Cek frasa negasi negatif (Bahasa Indonesia & English)
    if re.search(r'\b(tidak|kurang|bukan|belum|ga|gak|nggak|not|never|barely|hardly|didnt|didn\'t|dont|don\'t|doesnt|doesn\'t)\s+(enak|mantap|manis|sedap|rapi|cepat|bersih|puas|ramah|dibalas|dibales|good|delicious|tasty|fresh|clean|fast|friendly|worth|satisfied|polite|proper|hot|warm|cooked)\b', preprocessed_text):
        neg_score += 2.5

    # Cek frasa keluhan tajam (Bahasa Indonesia & English)
    if re.search(r'\b(nyesel|kecewa|kapok|nipu|judes|jutek|zonk|hambar|basi|tumpah|regret|disappointed|disappointing|scam|scammer|ripoff|awful|terrible|horrible|spoiled|rotten|spilled|leaking|disgusting|gross|vomit)\b', preprocessed_text):
        neg_score += 2.0

    # Cek frasa penguat positif (Bahasa Indonesia & English)
    if re.search(r'\b(enak|mantap|lezat|puas|ramah|cepat|segar|estetik|good|delicious|tasty|fresh|friendly|fast|clean|aesthetic|recommended|satisfying|great|amazing|awesome|nice)\s+(banget|sekali|pisan|so\s+much|a\s+lot)\b|\b(so|very|really|super|extremely|highly|truly|absolutely)\s+(good|delicious|tasty|fresh|friendly|fast|clean|aesthetic|recommended|satisfying|great|amazing|awesome|nice)\b', preprocessed_text):
        pos_score += 2.5

    # Pertimbangan Rating (jika ada nilai rating 1-5 bintang dari pembeli)
    if rating is not None:
        if rating >= 4.0:
            pos_score += 1.5
        elif rating <= 2.5:
            neg_score += 1.5

    # Aspek yang dibahas
    aspects = extract_aspects(preprocessed_text)

    # Saran positif membangun tanpa keluhan negatif tajam -> masuk ke Kritik & Saran Positif
    if suggestion_score > 0 and neg_score == 0:
        pos_score += 1.0

    if pos_score >= neg_score:
        # Kritik & Saran Positif
        total = max(pos_score + neg_score + 0.1, 1.0)
        confidence = min(round(0.72 + (pos_score / total) * 0.26, 2), 0.99)
        category = CommentCategory.POSITIF.value
        sentiment = SentimentType.POSITIVE.value
        action = ModerationAction.DELIVERED_TO_DASHBOARD.value
        is_delivered = True
        reason = "Komentar berupa feedback apresiasi atau saran positif yang membangun. Diteruskan ke dashboard toko."
        badge = "Apresiasi & Rekomendasi"
    else:
        # Kritik & Saran Negatif
        total = max(pos_score + neg_score + 0.1, 1.0)
        confidence = min(round(0.72 + (neg_score / total) * 0.26, 2), 0.99)
        category = CommentCategory.NEGATIF.value
        sentiment = SentimentType.NEGATIVE.value
        action = ModerationAction.DELIVERED_TO_DASHBOARD.value
        is_delivered = True
        reason = "Komentar berisi kritik/keluhan santun tanpa bahasa kasar. Diteruskan ke dashboard toko untuk evaluasi perbaikan."
        badge = "Perlu Perhatian / Evaluasi"

    return {
        "comment_id": comment_id,
        "tenant_id": tenant_id,
        "original_text": raw_text,
        "cleaned_text": preprocessed_text,
        "category": category,
        "action": action,
        "is_delivered_to_dashboard": is_delivered,
        "confidence_score": confidence,
        "sentiment": sentiment,
        "aspects": aspects,
        "flagged_words": [],
        "moderation_reason": reason,
        "dashboard_badge": badge
    }


def moderate_batch(comments: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Moderasi dan klasifikasi kumpulan komentar sekaligus (Bilingual: ID & EN)."""
    processed_comments = []
    delivered_to_dashboard = []
    auto_removed = []

    summary_counts = {
        CommentCategory.POSITIF.value: 0,
        CommentCategory.NEGATIF.value: 0,
        CommentCategory.ANEH_KASAR.value: 0,
    }

    for item in comments:
        cid = item.get("id") or item.get("comment_id")
        tid = item.get("tenant_id") or item.get("tenantId")
        text = item.get("text") or item.get("comment") or item.get("comment_text") or ""
        rating = item.get("rating")

        result = classify_comment(text=text, comment_id=cid, tenant_id=tid, rating=rating)
        processed_comments.append(result)
        summary_counts[result["category"]] += 1

        if result["is_delivered_to_dashboard"]:
            delivered_to_dashboard.append(result)
        else:
            auto_removed.append(result)

    return {
        "total_processed": len(comments),
        "total_delivered_to_dashboard": len(delivered_to_dashboard),
        "total_auto_removed": len(auto_removed),
        "summary_by_category": summary_counts,
        "delivered_comments": delivered_to_dashboard,
        "removed_comments": auto_removed,
        "all_results": processed_comments
    }

print("✅ Core Bilingual NLU Classifier & Batch Moderator siap digunakan!")


✅ Core NLU Classifier & Batch Moderator siap digunakan!


In [7]:
# CELL 7 — Pydantic Schemas (Request & Response Model API)
from pydantic import BaseModel, Field

class CommentClassifyRequest(BaseModel):
    comment_text: str = Field(..., description="Teks ulasan pembeli untuk toko", example="Makanannya enak banget, porsinya pas!")
    comment_id: Optional[str] = Field(None, description="ID ulasan dari database", example="cmt-101")
    tenant_id: Optional[str] = Field(None, description="ID tenant/toko penjual", example="tenant-7")
    rating: Optional[float] = Field(None, ge=1.0, le=5.0, description="Rating bintang ulasan 1-5")


class CommentClassifyResponse(BaseModel):
    comment_id: Optional[str] = None
    tenant_id: Optional[str] = None
    original_text: str
    cleaned_text: str
    category: str = Field(..., description="Kategori klasifikasi ulasan")
    action: str = Field(..., description="Aksi: DELIVERED_TO_DASHBOARD atau AUTO_REMOVED")
    is_delivered_to_dashboard: bool = Field(..., description="Status apakah ulasan diteruskan ke dashboard penjual")
    confidence_score: float = Field(..., description="Skor kepercayaan model [0-1]")
    sentiment: str = Field(..., description="Sentimen ulasan: POSITIVE, NEGATIVE, TOXIC")
    aspects: List[str] = Field(default_factory=list, description="Aspek-aspek yang dibahas")
    flagged_words: List[str] = Field(default_factory=list, description="Kata-kata kasar yang terdeteksi")
    moderation_reason: str = Field(..., description="Alasan moderasi sistem")
    dashboard_badge: Optional[str] = Field(None, description="Badge UI untuk dashboard penjual")


class CommentBatchItem(BaseModel):
    comment_id: Optional[str] = None
    tenant_id: Optional[str] = None
    comment_text: str
    rating: Optional[float] = None


class CommentBatchModerateRequest(BaseModel):
    comments: List[CommentBatchItem]


class CommentBatchModerateResponse(BaseModel):
    total_processed: int
    total_delivered_to_dashboard: int
    total_auto_removed: int
    summary_by_category: Dict[str, int]
    delivered_comments: List[CommentClassifyResponse]
    removed_comments: List[CommentClassifyResponse]
    all_results: List[CommentClassifyResponse]

print("✅ Pydantic Schemas siap untuk validasi API!")


✅ Pydantic Schemas siap untuk validasi API!


In [8]:
# CELL 8 — FastAPI App + Jalankan Server di Notebook
import nest_asyncio
import uvicorn
import threading
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import RedirectResponse

nest_asyncio.apply()

app = FastAPI(
    title="Tenantify AI — NLU Comment Moderation API",
    description="Microservice klasifikasi ulasan toko berbasis NLU untuk memoderasi kata kasar dan meneruskan kritik positif/negatif ke dashboard.",
    version="1.0.0",
    docs_url="/docs"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/", include_in_schema=False)
def root():
    return RedirectResponse(url="/docs")

@app.post("/comments/classify", response_model=CommentClassifyResponse, tags=["Comment NLU Moderation"])
def api_classify_comment(req: CommentClassifyRequest):
    """Klasifikasi ulasan toko secara real-time."""
    result = classify_comment(
        text=req.comment_text,
        comment_id=req.comment_id,
        tenant_id=req.tenant_id,
        rating=req.rating
    )
    return CommentClassifyResponse(**result)

@app.post("/comments/moderate-batch", response_model=CommentBatchModerateResponse, tags=["Comment NLU Moderation"])
def api_moderate_batch(req: CommentBatchModerateRequest):
    """Batch moderasi banyak ulasan toko sekaligus."""
    raw_list = [
        {
            "id": c.comment_id,
            "tenant_id": c.tenant_id,
            "text": c.comment_text,
            "rating": c.rating
        }
        for c in req.comments
    ]
    batch_res = moderate_batch(raw_list)
    return CommentBatchModerateResponse(
        total_processed=batch_res["total_processed"],
        total_delivered_to_dashboard=batch_res["total_delivered_to_dashboard"],
        total_auto_removed=batch_res["total_auto_removed"],
        summary_by_category=batch_res["summary_by_category"],
        delivered_comments=[CommentClassifyResponse(**c) for c in batch_res["delivered_comments"]],
        removed_comments=[CommentClassifyResponse(**c) for c in batch_res["removed_comments"]],
        all_results=[CommentClassifyResponse(**c) for c in batch_res["all_results"]],
    )

@app.get("/comments/categories", tags=["Comment NLU Moderation"])
def api_categories():
    """Metadata 3 parameter klasifikasi ulasan."""
    return {
        "categories": [
            {
                "name": CommentCategory.POSITIF.value,
                "action": ModerationAction.DELIVERED_TO_DASHBOARD.value,
                "target": "Dashboard Toko / Penjual",
                "description": "Komentar apresiasi, pujian rasa/kemasan/pelayanan, atau saran positif yang membangun.",
                "badge": "Apresiasi & Rekomendasi"
            },
            {
                "name": CommentCategory.NEGATIF.value,
                "action": ModerationAction.DELIVERED_TO_DASHBOARD.value,
                "target": "Dashboard Toko / Penjual",
                "description": "Kritik atau keluhan santun mengenai kualitas makanan/pelayanan/pengiriman tanpa kata kasar untuk evaluasi toko.",
                "badge": "Perlu Perhatian / Evaluasi"
            },
            {
                "name": CommentCategory.ANEH_KASAR.value,
                "action": ModerationAction.AUTO_REMOVED.value,
                "target": "Otomatis Dihapus (Auto-Removed)",
                "description": "Komentar toxic yang mengandung makian, kata kasar, pelecehan, atau sensor leetspeak kasar.",
                "badge": "DIBLOKIR / REMOVED"
            }
        ]
    }

def run_server(port=8001):
    config = uvicorn.Config(app=app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    server.run()

# Jalankan server FastAPI di background thread pada port 8001
server_port = 8001
t = threading.Thread(target=run_server, args=(server_port,), daemon=True)
t.start()
time.sleep(1.5)

print(f"🚀 Server NLU Moderation berhasil berjalan di background!")
print(f"📖 Swagger Docs : http://127.0.0.1:{server_port}/docs")


🚀 Server NLU Moderation berhasil berjalan di background!
📖 Swagger Docs : http://127.0.0.1:8001/docs


In [9]:
# CELL 9 — Demo Pengujian 24 Skenario Nyata (Bilingual: Indonesia & English)

test_scenarios = [
    # ── Kategori: Kritik & Saran Positif (Bahasa Indonesia)
    ("Makanannya enak bgt, porsinya pas dan packing rapi!", "Kritik & Saran Positif"),
    ("Kopinya mantap, mungkin ke depannya bisa sediain varian oat milk ya kak", "Kritik & Saran Positif"),
    ("Saran saya packagingnya diganti paper bag ya biar lebih eco friendly", "Kritik & Saran Positif"),
    ("Pelayanan ramah banget, packagingnya rapi dan estetik.", "Kritik & Saran Positif"),
    ("Gila enak bgt parah nagih!", "Kritik & Saran Positif"),
    ("Makanannya ciamik pol, porsi melimpah murmer!", "Kritik & Saran Positif"),

    # ── Kategori: Kritik & Saran Positif (English)
    ("The food was super delicious, huge portion and prompt delivery!", "Kritik & Saran Positif"),
    ("Really loved the coffee, amazing aroma and friendly barista!", "Kritik & Saran Positif"),
    ("Neat packaging and aesthetic presentation, highly recommended!", "Kritik & Saran Positif"),
    ("Tasty burger, maybe in the future you could add more vegan options please", "Kritik & Saran Positif"),
    ("Awesome taste and fast delivery! Will definitely reorder soon.", "Kritik & Saran Positif"),

    # ── Kategori: Kritik & Saran Negatif (Bahasa Indonesia)
    ("Ayamnya masih mentah bagian dalam kak, tolong lebih teliti lagi masaknya", "Kritik & Saran Negatif"),
    ("Pesanan sampai 1 jam lebih, makanannya sudah dingin dan kuahnya tumpah", "Kritik & Saran Negatif"),
    ("Adminnya judes banget dichat ga dibales", "Kritik & Saran Negatif"),
    ("porsinya dikit banget min, nyesel beli mahal-mahal", "Kritik & Saran Negatif"),
    ("Rasa kuahnya hambar cemplang, ayamnya alot keras, mahal bener kapok", "Kritik & Saran Negatif"),

    # ── Kategori: Kritik & Saran Negatif (English)
    ("The chicken was still raw inside, please cook it more carefully next time", "Kritik & Saran Negatif"),
    ("Waited over 1 hour, the food arrived completely cold and the soup was spilled", "Kritik & Saran Negatif"),
    ("Very rude cashier and slow response, never replied to my chat", "Kritik & Saran Negatif"),
    ("Tiny portion for an expensive price, totally regret buying here", "Kritik & Saran Negatif"),
    ("The pasta was bland and oversalted, disappointing quality", "Kritik & Saran Negatif"),

    # ── Kategori: Comment aneh-aneh (Bahasa Indonesia)
    ("Woy babi lu jualan makanan basi anjing!", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("Penjual t0l0l, nyesel beli di toko sampah ini gblk", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("makanan baji***an lu nipu gue", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("b.a.b.i lu jualan kaga bener", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("4nj1nk lu gausah jualan", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("Jancuk lu jualan basi!", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("Woy k*nt*l pesanan gua mana?", "Comment aneh-aneh (komentar yang berbahasa kasar)"),

    # ── Kategori: Comment aneh-aneh (English)
    ("What the fuck is this food, you stupid seller!", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("This store is full of assholes, trash service!", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("F*ck you, scammer bitch!", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("Piece of shit food, go to hell moron", "Comment aneh-aneh (komentar yang berbahasa kasar)"),
    ("Disgusting sh*t, never order from this bastard again", "Comment aneh-aneh (komentar yang berbahasa kasar)")
]

print("=" * 85)
print("🛡️ HASIL PENGUJIAN AKURASI MODEL BILINGUAL NLU MODERATOR (TENANTIFY)")
print("=" * 85)

passed = 0
for text, expected in test_scenarios:
    res = classify_comment(text)
    actual = res["category"]
    action = res["action"]
    is_ok = (actual == expected)
    if is_ok:
        passed += 1
    
    icon = "✅" if is_ok else "❌"
    dash_status = "🟢 Masuk Dashboard" if res["is_delivered_to_dashboard"] else "🔴 Dihapus (Auto-Removed)"
    
    print(f"{icon} Input       : {text}")
    print(f"   Expected    : {expected}")
    print(f"   Hasil NLU   : {actual}")
    print(f"   Aksi Sistem : {action} ➔ {dash_status}")
    if res["aspects"]:
        print(f"   Aspek Topik : {', '.join(res['aspects'])}")
    if res["flagged_words"]:
        print(f"   Kata Kasar  : {', '.join(res['flagged_words'])}")
    print("-" * 85)

accuracy = (passed / len(test_scenarios)) * 100
print(f"\n🎯 TOTAL AKURASI BILINGUAL: {passed}/{len(test_scenarios)} ({accuracy:.1f}%)")


🛡️ HASIL PENGUJIAN AKURASI MODEL NLU MODERATOR (TENANTIFY)
✅ Input       : Makanannya enak bgt, porsinya pas dan packing rapi!
   Expected    : Kritik & Saran Positif
   Hasil NLU   : Kritik & Saran Positif
   Aksi Sistem : DELIVERED_TO_DASHBOARD ➔ 🟢 Masuk Dashboard
   Aspek Topik : Rasa & Kualitas, Porsi & Takaran, Kemasan & Kebersihan
-------------------------------------------------------------------------------------
✅ Input       : Kopinya mantap, mungkin ke depannya bisa sediain varian oat milk ya kak
   Expected    : Kritik & Saran Positif
   Hasil NLU   : Kritik & Saran Positif
   Aksi Sistem : DELIVERED_TO_DASHBOARD ➔ 🟢 Masuk Dashboard
   Aspek Topik : Rasa & Kualitas
-------------------------------------------------------------------------------------
✅ Input       : Saran saya packagingnya diganti paper bag ya biar lebih eco friendly
   Expected    : Kritik & Saran Positif
   Hasil NLU   : Kritik & Saran Positif
   Aksi Sistem : DELIVERED_TO_DASHBOARD ➔ 🟢 Masuk Dashboard
   

In [10]:
# CELL 10 — Demo Batch Moderasi & Simulasi Tampilan Dashboard Penjual

sample_tenant_batch = [
    {"id": "rev-01", "tenantId": "tenant-1", "comment": "Kopi susunya creamy banget, next adain opsi less sugar ya!", "rating": 5.0},
    {"id": "rev-02", "tenantId": "tenant-1", "comment": "Woy kontol kopi apaan ini kemanisan anjink", "rating": 1.0},
    {"id": "rev-03", "tenantId": "tenant-1", "comment": "Roti bakarnya agak gosong pinggirannya kak, tolong diperbaiki", "rating": 2.5},
    {"id": "rev-04", "tenantId": "tenant-1", "comment": "Pelayanan mas baristanya ramah banget!", "rating": 5.0},
    {"id": "rev-05", "tenantId": "tenant-1", "comment": "b.a.b.i lu nipu pesanan kurang 1", "rating": 1.0}
]

batch_summary = moderate_batch(sample_tenant_batch)

print("📊 RINGKASAN MODERASI ULASAN TOKO:")
print(f" • Total Ulasan Masuk        : {batch_summary['total_processed']}")
print(f" • Diteruskan ke Dashboard   : {batch_summary['total_delivered_to_dashboard']} (Layak Tampil)")
print(f" • Otomatis Dihapus (Kasar)  : {batch_summary['total_auto_removed']} (Auto-Removed)")
print("\nDistribusi Kategori:")
for cat, count in batch_summary['summary_by_category'].items():
    print(f" - {cat}: {count}")

print("\n" + "=" * 60)
print("🏪 SIMULASI TAMPILAN DI DASHBOARD TOKO (Hanya Ulasan Lolos): ")
print("=" * 60)
for item in batch_summary['delivered_comments']:
    print(f"[{item['dashboard_badge']}] {item['original_text']}")
    print(f"   Topik: {item['aspects']} | Confidence: {item['confidence_score']}")
    print("-" * 60)


📊 RINGKASAN MODERASI ULASAN TOKO:
 • Total Ulasan Masuk        : 5
 • Diteruskan ke Dashboard   : 3 (Layak Tampil)
 • Otomatis Dihapus (Kasar)  : 2 (Auto-Removed)

Distribusi Kategori:
 - Kritik & Saran Positif: 2
 - Kritik & Saran Negatif: 1
 - Comment aneh-aneh (komentar yang berbahasa kasar): 2

🏪 SIMULASI TAMPILAN DI DASHBOARD TOKO (Hanya Ulasan Lolos): 
[Apresiasi & Rekomendasi] Kopi susunya creamy banget, next adain opsi less sugar ya!
   Topik: [] | Confidence: 0.97
------------------------------------------------------------
[Perlu Perhatian / Evaluasi] Roti bakarnya agak gosong pinggirannya kak, tolong diperbaiki
   Topik: ['Rasa & Kualitas'] | Confidence: 0.97
------------------------------------------------------------
[Apresiasi & Rekomendasi] Pelayanan mas baristanya ramah banget!
   Topik: ['Pelayanan & Tenant'] | Confidence: 0.97
------------------------------------------------------------


In [12]:
# CELL 11 — (Opsional) Public URL Tunnel via Ngrok
# Jalankan cell ini jika ingin menguji endpoint dari Postman / Frontend eksternal

from pyngrok import ngrok


ngrok.kill()

public_url = ngrok.connect(8001)
clean_url = public_url.public_url

print(f" Public API URL : {clean_url}")
print(f" Swagger Docs : {clean_url}/docs")


t=2026-11-09T12:39:49+0700 lvl=warn msg="can't bind default web address, trying alternatives" obj=web addr=127.0.0.1:4040
t=2026-11-09T12:39:49+0700 lvl=warn msg="failed to start tunnel" pg=/api/endpoints id=cc65db0c0a28c901 err="failed to start tunnel: The endpoint 'https://encircle-useable-unkind.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://encircle-useable-unkind.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}
